# [Title]

## Preparation

- [Github link](google.com) *[Optional]*

- Number of words: ***

- Runtime: *** hours (*Memory 10 GB, CPU Intel i7-10700 CPU @2.90GHz*)

- Coding environment: SDS Docker (or anything else)

- License: this notebook is made available under the [Creative Commons Attribution license](https://creativecommons.org/licenses/by/4.0/) (or other license that you like).

- Additional library *[libraries not included in SDS Docker or not used in this module]*:
    - **watermark**: A Jupyter Notebook extension for printing timestamps, version numbers, and hardware information.
    - ......

## Table of contents

1. [Introduction](#Introduction)

1. [Research questions](#Research-questions)

1. [Data](#Data)

1. [Methodology](#Methodology)

1. [Results and discussion](#Results-and-discussion)

1. [Conclusion](#Conclusion)

1. [References](#References)

## Introduction

[[ go back to the top ]](#Table-of-contents)

In [ ]:
# 准备工作：导入本项目需要的库，并统一设置随机种子，保证结果可复现。
# 如果你的 notebook 环境缺少某些库，可先在单独单元运行：
# %pip install pandas numpy matplotlib seaborn scikit-learn geopandas shapely

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from urllib.parse import quote

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 设置图形风格，让后续图表在 coursework 中更清晰。
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

RANDOM_STATE = 42

# 数据统一从 GitHub 读取，便于 notebook 在干净环境中复现。
GITHUB_OWNER = "hou1020"
GITHUB_REPO = "CASA0006-coursework"
GITHUB_BRANCH = "main"
GITHUB_RAW_BASE = f"https://raw.githubusercontent.com/{GITHUB_OWNER}/{GITHUB_REPO}/{GITHUB_BRANCH}"
GITHUB_ARCHIVE_URL = f"https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}/archive/refs/heads/{GITHUB_BRANCH}.zip"

def github_raw_url(path):
    return f"{GITHUB_RAW_BASE}/{quote(path, safe='/')}"

recent_crime_url = github_raw_url("MPS Ward Level Crime (most recent 24 months).csv")
historical_crime_url = github_raw_url("MPS Ward Level Crime (Historical).csv")
profile_url = github_raw_url("ward-profiles-excel-version.csv")
ptai_url = github_raw_url("Ward2014 AvPTAI2015.csv")
ward_shp_zip_member = (
    f"{GITHUB_REPO}-{GITHUB_BRANCH}/"
    "London-wards-2014/London-wards-2014_ESRI/London_Ward_CityMerged.shp"
)


## Research questions

[[ go back to the top ]](#Table-of-contents)

In [ ]:
# 研究问题对应的机器学习任务：
# 使用 2014/15 ward profile 的社会人口与建成环境特征，预测该 ward 是否属于高记录犯罪风险区域。
# y = 1 表示 2014/15 犯罪率位于 matched profile wards 的前 25%；y = 0 表示其余 matched wards。

research_question = (
    "To what extent can socio-demographic and built environment features "
    "distinguish high recorded-crime risk wards from other wards in London?"
)
print(research_question)


## Data

[[ go back to the top ]](#Table-of-contents)

In [ ]:
# 1. 读取数据、统一年份与检查 ward 匹配
# 核心建模数据使用 ward profile 的 2014/15 crime rate，与大部分解释变量的 2011-2015 年份口径一致。
# 两个 MPS 月度 crime 文件仍读取进来，只用于 ward-code compatibility audit。
recent_crime_raw = pd.read_csv(recent_crime_url)
historical_crime_raw = pd.read_csv(historical_crime_url)
profiles_raw = pd.read_csv(profile_url, encoding="latin1")
ptai_raw = pd.read_csv(ptai_url)

print("Recent MPS crime data shape:", recent_crime_raw.shape)
print("Historical MPS crime data shape:", historical_crime_raw.shape)
print("Ward profile shape:", profiles_raw.shape)
print("PTAI data shape:", ptai_raw.shape)

def normalise_name(series):
    return (
        series.astype(str)
        .str.lower()
        .str.replace("&", "and", regex=False)
        .str.replace(r"[^a-z0-9]+", " ", regex=True)
        .str.strip()
    )

profiles = profiles_raw.drop_duplicates(subset="New code", keep="first").copy()
profiles["Profile_Borough"] = profiles["Ward name"].str.split(" - ", n=1).str[0]
profiles["Profile_WardName"] = profiles["Ward name"].str.split(" - ", n=1).str[1]
profiles["Profile_WardName"] = profiles["Profile_WardName"].fillna(profiles["Ward name"])
profiles["borough_key"] = normalise_name(profiles["Profile_Borough"])
profiles["ward_key"] = normalise_name(profiles["Profile_WardName"])

ptai = ptai_raw.drop_duplicates(subset="Ward Code", keep="first").copy()

def crime_profile_match_counts(crime_df, label):
    crime_codes = set(crime_df["WardCode"].dropna())
    profile_new_codes = set(profiles["New code"].dropna())
    profile_old_codes = set(profiles["Old code"].dropna())

    crime_wards = crime_df[["WardCode", "WardName", "LookUp_BoroughName"]].drop_duplicates().copy()
    crime_wards["borough_key"] = normalise_name(crime_wards["LookUp_BoroughName"])
    crime_wards["ward_key"] = normalise_name(crime_wards["WardName"])
    name_matches = crime_wards.merge(
        profiles[["borough_key", "ward_key", "New code"]],
        on=["borough_key", "ward_key"],
        how="inner",
    )["WardCode"].nunique()

    return {
        "source": label,
        "crime_ward_codes": len(crime_codes),
        "code_matches_to_profile_new_code": len(crime_codes & profile_new_codes),
        "code_matches_to_profile_old_code": len(crime_codes & profile_old_codes),
        "borough_name_matches_to_profile": name_matches,
    }

ward_audit_df = pd.DataFrame([
    crime_profile_match_counts(recent_crime_raw, "MPS recent monthly crime, 2025 ward codes"),
    crime_profile_match_counts(historical_crime_raw, "MPS historical monthly crime, recoded to 2025 ward codes"),
])
display(ward_audit_df)

print(
    "The MPS monthly files use 2025-style ward codes, which have zero direct code matches "
    "to the 2014/15 ward profile codes. For year/geography consistency, the model below "
    "therefore uses the profile's own Crime rate - 2014/15 as the target."
)

# 建模数据：2014/15 crime target + 2011-2015 explanatory variables on the same profile geography.
ward_df = profiles.merge(
    ptai[["Ward Code", "AvPTAI2015", "PTAL"]],
    left_on="New code",
    right_on="Ward Code",
    how="left",
)

n_profile_wards = profiles["New code"].nunique()
n_modelling_wards = ward_df["New code"].nunique()
print(f"Profile wards available for modelling: {n_modelling_wards} out of {n_profile_wards} unique profile ward codes")
print("PTAI matches:", ward_df["AvPTAI2015"].notna().sum(), "out of", len(ward_df))
print("Modelling data shape:", ward_df.shape)
display(ward_df[["New code", "Ward name", "Population - 2015", "Crime rate - 2014/15", "AvPTAI2015"]].head())


*[Note: a table that describes the selected variables for analysis and modelling is required - see the example below.]*

| Variable                            | Type         | Description                                                             |Notes   |
|-------------------------------------|--------------|-------------------------------------------------------------------------|---|
| Burglary crime rate                 | Numeric      | The burglary rate of MSOAs. Used as dependent variables in regression.  |   |
| Temperature                         | Numeric      | The daytime temperature                                                 |   |
| Indicator of Inner or Outer London  | Categorical  | Whether the MSOA is in Inner London.                                    |   |
| ......  | ......  | ......                                    |   |

## Methodology

[[ go back to the top ]](#Table-of-contents)

*[Note: a flow chart that describes the methodology is strongly encouraged - see the example below. This flow chart can be made using Microsoft powerpoint or visio or other software]*

Source: see [link](https://linkinghub.elsevier.com/retrieve/pii/S2210670722004437).

![image.png](attachment:image.png)

In [ ]:
# 2. 构建犯罪率与二分类目标变量
# 犯罪率直接使用 ward profile 中的 Crime rate - 2014/15，与人口、住房、交通等解释变量年份更一致。
# 这避免了把 2025 ward crime counts 强行匹配到 2014/15 profile geography。
analysis_df = ward_df.copy()
analysis_df = analysis_df.dropna(subset=["Crime rate - 2014/15", "Population - 2015"]).copy()
analysis_df["Annual_Crime_Rate"] = analysis_df["Crime rate - 2014/15"]
analysis_df["Estimated_Annual_Crime"] = (
    analysis_df["Annual_Crime_Rate"] * analysis_df["Population - 2015"] / 1000
)

# 将 2014/15 犯罪率前 25% 的 profile ward 标记为高风险区。
crime_rate_threshold = analysis_df["Annual_Crime_Rate"].quantile(0.75)
analysis_df["High_Risk"] = (analysis_df["Annual_Crime_Rate"] >= crime_rate_threshold).astype(int)

print(f"75th percentile 2014/15 crime-rate threshold: {crime_rate_threshold:.2f} crimes per 1,000 residents")
print("Class balance among profile wards:")
display(
    analysis_df["High_Risk"]
    .value_counts()
    .rename(index={0: "Low risk", 1: "High risk"})
    .to_frame("count")
    .assign(share=lambda x: x["count"] / x["count"].sum())
)

display(
    analysis_df[["New code", "Ward name", "Estimated_Annual_Crime", "Population - 2015", "Annual_Crime_Rate", "High_Risk"]]
    .sort_values("Annual_Crime_Rate", ascending=False)
    .head(10)
)


In [ ]:
# 3. 探索性分析：目标变量分布与关键变量关系
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(data=analysis_df, x="Annual_Crime_Rate", hue="High_Risk", bins=30, ax=axes[0])
axes[0].axvline(crime_rate_threshold, color="black", linestyle="--", label="75th percentile")
axes[0].set_title("Annual recorded crime-rate distribution")
axes[0].set_xlabel("Annual crime rate per 1,000 residents")
axes[0].legend()

sns.countplot(data=analysis_df, x="High_Risk", ax=axes[1])
axes[1].set_title("Binary target balance")
axes[1].set_xlabel("High risk label")
axes[1].set_xticklabels(["Low risk (0)", "High risk (1)"])
axes[1].set_ylabel("Number of profile wards")

plt.tight_layout()
plt.show()

# 变量字典：只保留一小组有理论依据、可解释的社会人口与建成环境变量。
# 避免把 ward profile 中 60+ 个变量全部塞进模型，降低过拟合和解释混乱的风险。
selected_feature_candidates = [
    "Population density (persons per sq km) - 2013",
    "% BAME - 2011",
    "% Not Born in UK - 2011",
    "Employment rate (16-64) - 2011",
    "Median Household income estimate (2012/13)",
    "% Flat, maisonette or apartment - 2011",
    "% Households Social Rented - 2011",
    "% Households Private Rented - 2011",
    "Claimant rate of key out-of-work benefits (working age client group) (2014)",
    "% with no qualifications - 2011",
    "% with Level 4 qualifications and above - 2011",
    "% area that is open space - 2014",
    "Cars per household - 2011",
    "AvPTAI2015",
]
selected_feature_candidates = [col for col in selected_feature_candidates if col in analysis_df.columns]

variable_dictionary = pd.DataFrame({
    "Variable": selected_feature_candidates,
    "Type": [str(analysis_df[col].dtype) for col in selected_feature_candidates],
    "Missing values": [int(analysis_df[col].isna().sum()) for col in selected_feature_candidates],
    "Description / Notes": "Selected socio-demographic or built environment feature",
})
print(f"Selected modelling features: {len(selected_feature_candidates)}")
display(variable_dictionary)


In [ ]:
# 4. 空间可视化：高风险 ward 等值线地图
# 使用 2014 London ward boundary，尽量与 profile 的 2014/15 target geography 保持一致。
# 如果环境未安装 geopandas，本单元会跳过地图；模型部分不受影响。
try:
    import geopandas as gpd
    from tempfile import TemporaryDirectory
    from urllib.request import urlretrieve

    with TemporaryDirectory() as tmpdir:
        archive_path = Path(tmpdir) / "coursework-data.zip"
        urlretrieve(GITHUB_ARCHIVE_URL, archive_path)
        wards_gdf = gpd.read_file(f"zip://{archive_path}!{ward_shp_zip_member}")

    possible_code_cols = ["GSS_CODE", "WD14CD", "WardCode", "New code"]
    code_col = next((col for col in possible_code_cols if col in wards_gdf.columns), None)
    if code_col is None:
        raise ValueError(f"Cannot find ward code column in shapefile. Columns: {wards_gdf.columns.tolist()}")

    profile_codes = set(analysis_df["New code"])
    boundary_codes = set(wards_gdf[code_col])
    boundary_profile_matches = len(profile_codes & boundary_codes)
    print(f"2014 boundary wards matching profile codes: {boundary_profile_matches} out of {len(profile_codes)} profile wards")

    missing_boundary_codes = sorted(profile_codes - boundary_codes)
    if missing_boundary_codes:
        print("Profile ward codes not found in this boundary file:", missing_boundary_codes[:10])

    map_gdf = wards_gdf.merge(
        analysis_df[["New code", "Ward name", "Annual_Crime_Rate", "High_Risk"]],
        left_on=code_col,
        right_on="New code",
        how="left",
    )
    mapped_label_count = int(map_gdf["High_Risk"].notna().sum())
    print(f"Map join labels available for: {mapped_label_count} out of {len(map_gdf)} displayed boundary wards")

    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    map_gdf.plot(
        column="High_Risk",
        cmap="Set1",
        linewidth=0.2,
        edgecolor="white",
        legend=True,
        missing_kwds={"color": "lightgrey", "label": "No profile/model label"},
        ax=ax,
    )
    ax.set_title("High recorded-crime risk wards in London, 2014/15")
    ax.axis("off")
    plt.show()
except Exception as err:
    print("Spatial map skipped because geopandas/shapefile reading is unavailable:", err)


In [ ]:
# 5. 建模数据准备
# 删除编码、名称、目标变量和潜在泄漏变量；只使用上一步选定的小特征集。
X = analysis_df[selected_feature_candidates].copy()
y = analysis_df["High_Risk"].copy()

# 删除全空列，避免预处理阶段出现无意义特征。
X = X.dropna(axis=1, how="all")

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

print("Number of selected features:", X.shape[1])
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features), categorical_features)

# 使用 stratify=y 保持训练集和测试集中的高/低风险比例一致。
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train class balance:")
print(y_train.value_counts(normalize=True).sort_index())

# 数值变量：中位数填补 + 标准化；类别变量：众数填补 + one-hot 编码。
# 这个结构参考 practical 中 Pipeline / ColumnTransformer 的写法，便于复现实验。
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features),
])


In [ ]:
# 6. 基线模型：逻辑回归
# class_weight="balanced" 用于处理 1:3 左右的类别不平衡；模型系数也较容易解释。
log_reg_model = Pipeline([
    ("preprocess", preprocessor),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )),
])

log_reg_model.fit(X_train, y_train)
log_reg_pred = log_reg_model.predict(X_test)

print("Logistic Regression classification report:")
print(classification_report(y_test, log_reg_pred, target_names=["Low risk", "High risk"]))


In [ ]:
# 7. 进阶模型：随机森林分类器
# 随机森林可捕捉非线性关系；class_weight="balanced" 强化少数类（高风险 ward）的学习权重。
rf_model = Pipeline([
    ("preprocess", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=500,
        min_samples_leaf=3,
        max_features="sqrt",
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
])

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

print("Random Forest classification report:")
print(classification_report(y_test, rf_pred, target_names=["Low risk", "High risk"]))


## Results and discussion

[[ go back to the top ]](#Table-of-contents)

In [ ]:
# 8. 模型评估与比较
# 本研究更重视 High risk 的 recall，因为漏判高风险 ward 会影响资源预防性配置。
def collect_metrics(name, y_true, y_pred):
    return {
        "Model": name,
        "Precision_high_risk": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "Recall_high_risk": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "F1_high_risk": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
    }

results_df = pd.DataFrame([
    collect_metrics("Logistic Regression", y_test, log_reg_pred),
    collect_metrics("Random Forest", y_test, rf_pred),
]).sort_values("Recall_high_risk", ascending=False)

print("Single stratified train-test split performance:")
display(results_df)

# 加入 StratifiedKFold 交叉验证，减少单次 train-test split 对随机种子的依赖。
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scoring = {
    "precision_high_risk": "precision",
    "recall_high_risk": "recall",
    "f1_high_risk": "f1",
}

cv_rows = []
for name, model in [
    ("Logistic Regression", log_reg_model),
    ("Random Forest", rf_model),
]:
    cv_scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=cv_scoring,
        n_jobs=-1,
        error_score="raise",
    )
    cv_rows.append({
        "Model": name,
        "CV_precision_high_risk_mean": cv_scores["test_precision_high_risk"].mean(),
        "CV_precision_high_risk_std": cv_scores["test_precision_high_risk"].std(),
        "CV_recall_high_risk_mean": cv_scores["test_recall_high_risk"].mean(),
        "CV_recall_high_risk_std": cv_scores["test_recall_high_risk"].std(),
        "CV_f1_high_risk_mean": cv_scores["test_f1_high_risk"].mean(),
        "CV_f1_high_risk_std": cv_scores["test_f1_high_risk"].std(),
    })

cv_results_df = pd.DataFrame(cv_rows).sort_values("CV_recall_high_risk_mean", ascending=False)
print("5-fold StratifiedKFold cross-validation performance:")
display(cv_results_df)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, name, pred in zip(
    axes,
    ["Logistic Regression", "Random Forest"],
    [log_reg_pred, rf_pred],
):
    ConfusionMatrixDisplay.from_predictions(
        y_test,
        pred,
        display_labels=["Low risk", "High risk"],
        cmap="Blues",
        ax=ax,
        colorbar=False,
    )
    ax.set_title(name)
plt.tight_layout()
plt.show()


In [ ]:
# 9. 模型解释：逻辑回归系数与随机森林特征重要性
# get_feature_names_out() 可以把 ColumnTransformer 后的特征名取回，方便解释模型。
feature_names = log_reg_model.named_steps["preprocess"].get_feature_names_out()
clean_feature_names = [name.replace("num__", "").replace("cat__", "") for name in feature_names]

coef_df = pd.DataFrame({
    "feature": clean_feature_names,
    "coefficient": log_reg_model.named_steps["model"].coef_[0],
})
coef_df["abs_coefficient"] = coef_df["coefficient"].abs()
coef_top = coef_df.sort_values("abs_coefficient", ascending=False).head(12)

display(coef_top)

plt.figure(figsize=(10, 6))
sns.barplot(data=coef_top, y="feature", x="coefficient", palette="vlag")
plt.axvline(0, color="black", linewidth=1)
plt.title("Largest logistic regression coefficients")
plt.xlabel("Coefficient: positive values increase high-risk probability")
plt.ylabel("")
plt.tight_layout()
plt.show()

rf_importance_df = pd.DataFrame({
    "feature": clean_feature_names,
    "importance": rf_model.named_steps["model"].feature_importances_,
}).sort_values("importance", ascending=False)
rf_top = rf_importance_df.head(12)

display(rf_top)

plt.figure(figsize=(10, 6))
sns.barplot(data=rf_top, y="feature", x="importance", color="#4C78A8")
plt.title("Random Forest feature importance")
plt.xlabel("Mean decrease in impurity")
plt.ylabel("")
plt.tight_layout()
plt.show()


## Conclusion

[[ go back to the top ]](#Table-of-contents)

In [ ]:
# 10. 自动生成结果摘要，辅助撰写 Results / Conclusion 部分。
best_model_row = cv_results_df.iloc[0]
print("Model comparison summary")
print("------------------------")
print(f"Best model by cross-validated high-risk recall: {best_model_row['Model']}")
print(f"CV high-risk recall: {best_model_row['CV_recall_high_risk_mean']:.3f} ± {best_model_row['CV_recall_high_risk_std']:.3f}")
print(f"CV high-risk precision: {best_model_row['CV_precision_high_risk_mean']:.3f} ± {best_model_row['CV_precision_high_risk_std']:.3f}")
print(f"CV high-risk F1-score: {best_model_row['CV_f1_high_risk_mean']:.3f} ± {best_model_row['CV_f1_high_risk_std']:.3f}")
print()
print("Interpretation note:")
print(
    "In this resource-allocation framing, false negatives are especially costly because "
    "they represent high-risk wards that the model fails to identify. Therefore recall "
    "for the High risk class should be discussed alongside precision and F1-score. "
    "The modelling evidence applies to the matched ward subset with available profile features."
)


## References

[[ go back to the top ]](#Table-of-contents)

In [ ]:
# 参考文献记录建议：
# 1. 在正文中引用 Metropolitan Police Service crime data。
# 2. 引用 London ward profile / Census-derived socio-demographic variables。
# 3. 引用 scikit-learn 文档中 LogisticRegression、RandomForestClassifier、classification_report 等方法。
# 本单元不需要运行；保留为写作提醒。
